# Advanced AI-Driven Search
This notebook is a companion to the article Advanced AI-Driven Search demonstrates use of advanced techniques that can be leveraged to improve the performance of traditional RAG architrecture. </br>
Specifically, it covers four approaches: **learned sprase retrieval**, **mixture of encoders**, **wormhole vectors** and **relevance feedback**. </br>
Each of these techniques is explained and visualized. On top of that, there is always resuable code block that you can leverage in your application.

In [1]:
%cd ../

/Users/jakubzovakfilevine/Desktop/advanced_ai_driven_search


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from typing import cast

from datasets import load_dataset
from datasets.dataset_dict import DatasetDict
from datasets.dataset_dict import Dataset
from qdrant_client import QdrantClient
from qdrant_client.models import models
from qdrant_client.http.models.models import QueryResponse
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding
from fastembed.sparse.sparse_embedding_base import SparseEmbedding

from notebooks import (
    build_doc_vector,
    build_query_vector,
    display_retrieval_result,
)

## Dataset
For the demonstrations of the advanced concepts we will use modified version of the MS Macro dataset. <br>
Load the data from the Hugging Face dataset from [Zovi3/pa195_semestral_assignment](https://huggingface.co/datasets/Zovi3/pa195_semestral_assignment/upload/main). 

In [4]:
query_dataset_dict: DatasetDict = cast(DatasetDict, load_dataset(
    "Zovi3/pa195_semestral_assignment",
    data_dir="query-all-MiniLM-L6-v2-100-filters-embedded-results"
)) 
query_dataset: Dataset = query_dataset_dict["train"]
query_dataset

Dataset({
    features: ['text', 'id', 'filters', 'embedding', 'multi_vector_embedding', 'result'],
    num_rows: 100
})

In [5]:
# Locally prepared corpus (dense + multi-vector + precomputed SPLADE + rating).
# Produced by notebooks/prepare_dataset.ipynb and saved under ./data/.
NUM_THOUSANDS = 1
DOCUMENTS_DATASET_NAME = (
    f"corpus-all-MiniLM-L6-v2-{NUM_THOUSANDS}K-multi-vector-splade-metadata"
)

documents_dataset_dict: DatasetDict = DatasetDict.load_from_disk(
    f"./data/{DOCUMENTS_DATASET_NAME}"
)

documents: Dataset = documents_dataset_dict["train"]
documents

Dataset({
    features: ['text', 'id', 'embedding', 'multi_vector_embedding', 'splade_indices', 'splade_values', 'rating'],
    num_rows: 1000
})

## Setup

Define constatnts to be used later.

In [6]:
COLLECTION_NAME = "ms_macro"
DENSE_VECTOR_FIELD = "dense"
BM25_VECTOR_FIELD = "bm25"
SPLADE_VECTOR_FIELD = "splade"
MULTI_VECTOR_FIELD = "multi"
METADATA_AWARE_VECTOR_FIELD = "metadata_aware"

QUERY_EXAMPLE = "what is the study of virology"

Define models we are going to use throughout the notebook.

In [7]:
embedding_model = TextEmbedding('sentence-transformers/all-MiniLM-L6-v2')
embedding_model_size = 384

BM25_MODEL_NAME = "Qdrant/bm25"
bm25_model = SparseTextEmbedding(BM25_MODEL_NAME)

SPLADE_MODEL_NAME = "prithivida/Splade_PP_en_v1"
splade_model = SparseTextEmbedding(SPLADE_MODEL_NAME)

multi_vector_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")
multi_vector_model_size = 128

Initiate the Qdrant in memory instance.

In [8]:
client = QdrantClient(":memory:")

Create Qdrant collection to use for demonstration of different search types. Specifics of this code will be explained in later sections. 

In [9]:
try:
    client.delete_collection(COLLECTION_NAME)
except: 
    print(f"Collection {COLLECTION_NAME} does not exist. Creating new one.")
    
collection_created = client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        DENSE_VECTOR_FIELD: models.VectorParams(
            size=embedding_model_size,
            distance=models.Distance.COSINE
        ),
        MULTI_VECTOR_FIELD: models.VectorParams(
            distance=models.Distance.COSINE,
            size=multi_vector_model_size,
            multivector_config=models.MultiVectorConfig( # Configure max-sim operator
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            hnsw_config=models.HnswConfigDiff(m=0), # Disable HNSW index reranking
        ),
        METADATA_AWARE_VECTOR_FIELD: models.VectorParams(
            size=embedding_model_size + 3, # dense embedding + 3D rating encoding
            distance=models.Distance.COSINE,
        ),
    },
    sparse_vectors_config={
        BM25_VECTOR_FIELD: models.SparseVectorParams(modifier=models.Modifier.IDF),
        SPLADE_VECTOR_FIELD: models.SparseVectorParams()
    }
)

if collection_created:
    print(f"Created collection '{COLLECTION_NAME}'.")

rating_weight = 0.5
points = [
    models.PointStruct(
        id=doc["id"],
        vector={
            DENSE_VECTOR_FIELD: doc["embedding"],
            MULTI_VECTOR_FIELD: doc["multi_vector_embedding"],
            METADATA_AWARE_VECTOR_FIELD: build_doc_vector(
                doc["embedding"], doc["rating"], rating_weight=rating_weight
            ).tolist(),
            BM25_VECTOR_FIELD: models.Document(
                    text=doc["text"], model=BM25_MODEL_NAME

            ),
            SPLADE_VECTOR_FIELD: models.SparseVector(
                    indices=doc["splade_indices"], values=doc["splade_values"]
            ),
        },
        payload={"text": doc["text"], "rating": doc["rating"]},
    )
    for doc in documents
]
client.upload_points(collection_name=COLLECTION_NAME, points=points, batch_size=128)
print(f"Collection info: {client.get_collection(COLLECTION_NAME).points_count} points in collection")

Created collection 'ms_macro'.
Collection info: 1000 points in collection


## Basic Retrieval Augmented Generation (RAG)
<table>
  <tr>
    <td style="width: 50%; vertical-align: top; padding-right: 16px;">
      <p>
        RAG is a standard design pattern used in almost all AI applications that require custom knowledge base.
        There are two main components in RAG: retrieval and generation. In this notebook we solely focus on retrieval.
      </p>
      <p>
        A RAG pipeline is visualized in Figure 1. It uses a <strong>vector database</strong> for storage and retrieval of embedded documents,
        includes a re-ranking step, and then passes the retrieved documents to an LLM to generate the final answer.
      </p>
      <p>
The retrieval is done based on similarity which is defined by the distance function, in practice it is almost always cosine distance:

$$
\mathrm{d}(x,y) = 1 - \cos(x,y) = 1 - \frac{x \cdot y}{\|x\| \|y\|}
$$

In this notebook we use [Qdrant](https://github.com/qdrant/qdrant) as our vector database. It offers many advanced features like relevance feedback, multi-vector support..
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="https://raw.githubusercontent.com/Zovi343/wormhole_vectors/main/images/rag_pipeline.png" alt="Diagram of a RAG pipeline" width="500" />
      <p><b>Figure 1. RAG pipeline</b></p>
    </td>
  </tr>
</table>

#### Vector Search Query

In [10]:
query_dense_embedding: list[float] = list(embedding_model.query_embed(QUERY_EXAMPLE))[0].tolist()

retrieval_results: QueryResponse = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_dense_embedding,
        using="dense",
        limit=5
)

display_retrieval_result(retrieval_results)

SCORE    RATING  TEXT
-----------------------------------------------------------------
0.305    2       Etymology: L, vas + constrigere. 1 adj, pertaining...
0.271    5       A Systematic Approach to Care Planning: A step-by-...
0.259    4       1 A clear picture of the broad problem(s) or resea...
0.240    5       It is used to present the main points or topics of...
0.238    2       The project's broad objective was to improve patie...


## Sparse Retrieval
<table>
  <tr>
    <td style="width: 50%; vertical-align: top; padding-right: 16px;">
      <p>
        Hybrid-search is a natural step for improving basic RAG pipeline.
      </p>
      <p>
        BM25 algorithm is a default approach when it comes to the sparse retrieval. <br>
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="../images/hybrid_search.png" alt="Diagram of a Hybrid Retrieval" width="450" />
      <p><b>RAG pipeline</b></p>
    </td>
  </tr>
</table>

### Hybrid Search Query

In [11]:
query_dense_embedding: list[float] = list(embedding_model.query_embed(QUERY_EXAMPLE))[0].tolist()

sparse_limit = 10
dense_limit = 10
prefetch_sparse_and_dense_search: list[models.Prefetch] = [
        models.Prefetch(
                query=models.Document(
                        text=QUERY_EXAMPLE,
                        model=BM25_MODEL_NAME
                ),
                using=BM25_VECTOR_FIELD,
                limit=sparse_limit,
        ),
        models.Prefetch(
                query=query_dense_embedding,
                using=DENSE_VECTOR_FIELD,
                limit=dense_limit,
        ),
]

retrieval_results: QueryResponse = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=prefetch_sparse_and_dense_search,
        query=models.RrfQuery(rrf=models.Rrf(k=60)),
        limit=5
)

display_retrieval_result(retrieval_results)

SCORE    RATING  TEXT
-----------------------------------------------------------------
0.017    4       Fellowship recipients receive full-tuition for the...
0.017    2       Etymology: L, vas + constrigere. 1 adj, pertaining...
0.016    5       2/10/2001 @ Old School Skate Jam 2001. Since her s...
0.016    5       A Systematic Approach to Care Planning: A step-by-...
0.016    2       1 IC&RC has endorsed the new IC&RC Alcohol and Dru...


## Learned Sparse Retrieval
<table>
  <tr>
    <td style="width: 50%; vertical-align: top; padding-right: 16px;">
      <p>
        Hybrid-search is a natural step for improving basic RAG pipeline.
      </p>
      <p>
        TODO
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="../images/hybrid_search.png" alt="Diagram of a Hybrid Retrieval" width="300" />
      <p><b>Learned Sparse Retrieval</b></p>
    </td>
  </tr>
</table>

#### Splade
<table>
  <tr>
    <td style="width: 50%; vertical-align: top; padding-right: 16px;">
      <p>
        Splade is lorem ipsum lorem ipsum
      </p>
      <p>
        BM25 model often stragles with synonyms, homnonyms and other subtle relations in the syntax. <br>
        Splade model addresses by performing query espansion and also enrichnig sparse representation with semantic information.
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="../images/splade.png" alt="Diagram of a Hybrid Retrieval" width="250" />
      <p><b>SPLADE</b></p>
    </td>
  </tr>
</table>

### Hybrid Search Query With Splade

In [12]:
query_dense_embedding: list[float] = list(embedding_model.query_embed(QUERY_EXAMPLE))[0].tolist()

sparse_limit = 10
dense_limit = 10
prefetch_sparse_and_dense_search: list[models.Prefetch] = [
        models.Prefetch(
                query=models.Document(
                        text=QUERY_EXAMPLE,
                        model=SPLADE_MODEL_NAME
                ),
                using=SPLADE_VECTOR_FIELD,
                limit=sparse_limit,
        ),
        models.Prefetch(
                query=query_dense_embedding,
                using=DENSE_VECTOR_FIELD,
                limit=dense_limit,
        ),
]

retrieval_results: QueryResponse = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=prefetch_sparse_and_dense_search,
        query=models.RrfQuery(rrf=models.Rrf(k=60)),
        limit=5
)

display_retrieval_result(retrieval_results)

SCORE    RATING  TEXT
-----------------------------------------------------------------
0.031    2       The environment inside the winery is carefully con...
0.017    1       In the USA, founded in 1964, the National Kidney F...
0.017    2       Etymology: L, vas + constrigere. 1 adj, pertaining...
0.016    5       A Systematic Approach to Care Planning: A step-by-...
0.016    3       In broad usage, the terms armed forces and  milita...


## Metadata Aware Vector Search

### Metadata Boosting
<table>
  <tr>
    <td style="width: 50%; vertical-align: top; padding-right: 16px;">
      <p>
        Metadata boosting is lorem ipsum lorem ipsum
      </p>
      <p>
        TODO
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="../images/metadata_boosting.png" alt="Diagram of a Hybrid Retrieval" width="400" />
      <p><b>Metadata Boosting</b></p>
    </td>
  </tr>
</table>

In [13]:
query_dense_embedding: list[float] = list(embedding_model.query_embed(QUERY_EXAMPLE))[0].tolist()

dense_limit = 10
prefetch_sparse_and_dense_search: list[models.Prefetch] = [
        models.Prefetch(
                query=query_dense_embedding,
                using=DENSE_VECTOR_FIELD,
                limit=dense_limit,
        ),
]


rating_boost_weight = 0.05
retrieval_results: QueryResponse = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=prefetch_sparse_and_dense_search,
        query=models.FormulaQuery(
            formula=models.SumExpression(sum=[
                "$score",
                # Boost each document by its rating (1-5) scaled by the weight
                models.MultExpression(mult=[rating_boost_weight, "rating"]),
            ]
        )),
        limit=5
)

display_retrieval_result(retrieval_results)

SCORE    RATING  TEXT
-----------------------------------------------------------------
0.521    5       A Systematic Approach to Care Planning: A step-by-...
0.490    5       It is used to present the main points or topics of...
0.459    4       1 A clear picture of the broad problem(s) or resea...
0.433    4       Warning. Some species of bacteria are pathogenic, ...
0.405    2       Etymology: L, vas + constrigere. 1 adj, pertaining...


### Mixture of Encoders
<table>
  <tr>
    <td style="width: 50%; vertical-align: top; padding-right: 16px;">
      <p>
        Splade is lorem ipsum lorem ipsum
      </p>
      <p>
        TODO
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="../images/mixture_of_encoders.png" alt="Diagram of a Hybrid Retrieval" width="400" />
      <p><b>Mixture of Encoders</b></p>
    </td>
  </tr>
</table>

In [14]:
query_metadata_aware = build_query_vector(
    query_dense_embedding, rating_weight=rating_weight
).tolist()

retrieval_results: QueryResponse = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_metadata_aware,
        using=METADATA_AWARE_VECTOR_FIELD,
        limit=5
)

display_retrieval_result(retrieval_results)

SCORE    RATING  TEXT
-----------------------------------------------------------------
0.380    5       A Systematic Approach to Care Planning: A step-by-...
0.358    4       1 A clear picture of the broad problem(s) or resea...
0.358    5       It is used to present the main points or topics of...
0.339    5       The story of Civil War medicine is only less depre...
0.338    4       Warning. Some species of bacteria are pathogenic, ...


### Multi-Vector Model

<table>
  <tr>
    <td style="width: 50%; vertical-align: top;">
      <p>
        As mentioned above, we will implement re-ranking with multi-vector late interaction embeddings based on
        <a href="https://dl.acm.org/doi/epdf/10.1145/3397271.3401075">ColBERT</a>.
      </p>
      <p>
        In traditional encoder models such as BERT, the final vector is obtained by pooling over token embeddings. <br>
        In the late interaction approach, we skip pooling and instead use token-level embeddings to represent documents and queries. 
      </p>
      <p>
        This yields a much more expressive representation, usually at the cost of higher memory usage and query latency. <br>
        That is why late interaction is commonly used as a re-ranking step instead of the main retrieval stage.
      </p>
      <p>
      Late interaction can be leveraged as a re-ranking step with the use of <> the following distance function, called the MaxSim operator:

$$
\text{MaxSim}(q, D) = \sum_{i \in q} \max_{j \in D} \; \text{sim}(q_i, d_j)
$$

The MaxSim operator computes cosine similarity between each query token and each document token and then takes the maximum similarity for each query token. <br> It is visualized in **Figure 3**.
</p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="https://raw.githubusercontent.com/Zovi343/wormhole_vectors/main/images/max_sim.png" alt="Late Interaction With MaxSim Operator" width="500" />
      <p><b>Late Interaction With MaxSim Operator</b></p>
    </td>
  </tr>
</table>

### Multi-vector Search
It is general good practice to include reranking model in the IR system. <br>
Reranking uses stronger model to select the most relevant documents from the initial retrieval. <br>
You will implement reranking with multi-vector late interaction embedding ColBERT.

In [15]:
# Multi-vector search: cheap dense retrieval first, then re-rank the
# candidates with the ColBERT late-interaction (MaxSim) multi-vector.
query_multi_vector: list[list[float]] = list(multi_vector_model.query_embed(QUERY_EXAMPLE))[0].tolist()

rerank_prefetch_limit = 20
retrieval_results: QueryResponse = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_multi_vector,
        using=MULTI_VECTOR_FIELD,
        limit=5
)

display_retrieval_result(retrieval_results)

SCORE    RATING  TEXT
-----------------------------------------------------------------
16.796   1       In the USA, founded in 1964, the National Kidney F...
12.893   3       In broad usage, the terms armed forces and  milita...
11.967   2       The environment inside the winery is carefully con...
11.449   4       Scientists are studying bats and their use of echo...
10.896   5       Symbicort contains a combination of budesonide and...


### Wormhole Vectors

<table>
  <tr>
    <td style="width: 50%; vertical-align: top;">
      <p>
        As mentioned above, we will implement re-ranking with multi-vector late interaction embeddings based on
        <a href="https://dl.acm.org/doi/epdf/10.1145/3397271.3401075">ColBERT</a>.
      </p>
      <p>
        In traditional encoder models such as BERT, the final vector is obtained by pooling over token embeddings. <br>
        In the late interaction approach, we skip pooling and instead use token-level embeddings to represent documents and queries. 
      </p>
    </td>
    <td style="width: 50%; vertical-align: top; text-align: center;">
      <img src="../images/wormhole_vectors.png" alt="Wormhole Vectors" width="500" />
      <p><b>Wormhole Vectors</b></p>
    </td>
  </tr>
</table>

In [ ]:
# TODO: Add Wormhole Vectors